# End-to-End ETL Pipeline: Excel to PostgreSQL

This project demonstrates a structured ETL pipeline built using Python (Pandas) and PostgreSQL (Supabase).

The pipeline:
- Extracts structured data from Excel files
- Performs data cleaning and transformation
- Loads cleaned data into PostgreSQL
- Enforces relational integrity using Primary and Foreign Keys

## 1. Setup & Imports

In [1]:
import pandas as pd 
import sqlalchemy as db 
from sqlalchemy import create_engine

## 2. Database Connection

In [ ]:
# Create PostgreSQL engine (Supabase)
engine = create_engine("your_connection_string_here")

## 3. Customers ETL

In [ ]:
def etl_customers(file_path, engine_connection):
    
    # Extract
    customers = pd.read_excel(file_path)
    
    # Transform
    customers["Zipcode"] = customers["Zipcode"].astype(str)
    customers = customers.drop_duplicates(subset="Email", keep="first")
    
    # Load (Full Refresh)
    with engine_connection.begin() as conn:
        conn.execute("TRUNCATE TABLE customers CASCADE;")
    
    customers.to_sql(
        name="customers",
        con=engine_connection,
        if_exists="append",
        index=False
    )
    
    print("Customers ETL completed successfully.")

In [4]:
etl_customers('C:\\Users\\akash\\OneDrive\\Desktop\\Python\\ETL\\Chapter_2\\H+ Sport Customers.xlsx', engine)

C:\Users\akash\AppData\Local\Temp\ipykernel_26860\3737365646.py:13: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  conn.execute("TRUNCATE TABLE customers CASCADE;")


Customers ETL completed successfully.


## 4. Orders ETL

In [5]:
def etl_orders(file_path, engine_connection):

    # Extract
    orders = pd.read_excel(file_path)

    # Transform
    for col in ["CustomersComment", "SalespersonsComment"]:
        if col in orders.columns:
            orders.drop(columns=[col], inplace=True)

    orders["Date"] = pd.to_datetime(orders["Date"], errors="coerce")

    if orders["TotalDue"].dtype == object:
        orders["TotalDue"] = orders["TotalDue"].str.replace("$", "", regex=False)
    orders["TotalDue"] = pd.to_numeric(orders["TotalDue"], errors="coerce")

    # Load
    with engine_connection.connect() as conn:
        conn.execute("TRUNCATE TABLE orders;")
        
    orders.to_sql(name="orders", con=engine_connection, if_exists="append", index=False)

    print("Orders ETL completed successfully.")

In [6]:
etl_orders("C:\\Users\\akash\\OneDrive\\Desktop\\Python\\ETL\\Chapter_2\\H+ Sport Orders.xlsx", engine)

Orders ETL completed successfully.


## 5. Employees ETL

In [7]:
def etl_employees(file_path, engine_connection):

    # Extract
    employees = pd.read_excel(file_path, sheet_name="Employees-Table")

    # Transform
    for col in ["New Salary", "Tax Rate", "2.91%"]:
        if col in employees.columns:
            employees.drop(columns=[col], inplace=True)

    if "Benefits" in employees.columns:
        employees["Benefits"] = employees["Benefits"].fillna("No Benefits")

    employees = employees.drop_duplicates(subset=["Employee Name", "Hire Date"])

    employees["EmployeeID"] = range(1, len(employees) + 1)

    # Load
    with engine_connection.connect() as conn:
        conn.execute("TRUNCATE TABLE employees;")
        
    employees.to_sql(name="employees", con=engine_connection, if_exists="append", index=False)

    print("Employees ETL completed successfully.")



In [8]:
etl_employees("C:\\Users\\akash\\OneDrive\\Desktop\\Python\\ETL\\Chapter_2\\H+ Sport Employees.xlsx", engine)

Employees ETL completed successfully.


## Relational Integrity

Primary Keys:
- customers(CustomerID)
- orders(OrderID)

Foreign Key:
- orders(CustomerID) → customers(CustomerID)

The pipeline uses a full-refresh strategy with TRUNCATE + append,
while preserving relational constraints.